In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, time, sqlite3
import pandas as pd

PROJECT_ROOT = '/content/drive/MyDrive/Equity'
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
NOTEBOOKS_DIR = os.path.join(PROJECT_ROOT, 'notebooks')
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, 'outputs')
CHARTS_DIR = os.path.join(OUTPUTS_DIR, 'charts')
DB_PATH = os.path.join(DATA_DIR, 'equity_market.db')

max_wait = 30
waited = 0
while not os.path.exists(DB_PATH) and waited < max_wait:
    time.sleep(2)
    waited += 2
    print(f"Waiting for Drive to sync... ({waited}s)")

if not os.path.exists(DB_PATH):
    raise RuntimeError("DB_PATH not found after waiting. Check Drive mount manually before proceeding.")

size_mb = os.path.getsize(DB_PATH) / (1024 * 1024)
if size_mb < 1:
    raise RuntimeError(f"DB_PATH exists but is only {size_mb:.2f} MB. This is not the real database. STOP.")

print(f"Database verified: {size_mb:.2f} MB.")

Mounted at /content/drive
Database verified: 145.80 MB.


In [ ]:
conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
print("Connected in READ-ONLY mode.")

try:
    pd.read_sql("CREATE TABLE test_write_check (x INTEGER)", conn)
    print("WARNING: write succeeded — read-only protection is NOT working")
except Exception as e:
    print(f"Confirmed read-only: write attempt correctly failed with: {type(e).__name__}")

Connected in READ-ONLY mode.
Confirmed read-only: write attempt correctly failed with: DatabaseError


In [ ]:
!pip install -q anthropic streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 929.8/929.8 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 104.4 MB/s eta 0:00:00


In [ ]:
!pip install -q transformers accelerate torch

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

print("Downloading model (this happens once per session, ~3GB)...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float32)

print("Model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded.


In [ ]:
def ask_local_llm(prompt, max_new_tokens=300):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt")

    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.3, do_sample=True)
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response

test = ask_local_llm("Reply with exactly: connection successful")
print(test)

connection successful


In [ ]:
import re

def extract_sql(model_output):
    match = re.search(r"```sql\s*(.*?)```", model_output, re.DOTALL)
    if match:
        return match.group(1).strip()
    match = re.search(r"```\s*(.*?)```", model_output, re.DOTALL)
    if match:
        return match.group(1).strip()
    return None

def is_safe_query(sql):
    forbidden = ["insert", "update", "delete", "drop", "alter", "create", "attach", "pragma"]
    sql_lower = sql.lower()
    return not any(word in sql_lower for word in forbidden)

In [ ]:
SIMPLE_SYSTEM_PROMPT = """You are a financial data analyst assistant with access to a SQLite database.

Tables (use these EXACT column names, do not invent or rename any column):
1. daily_prices (ticker, date, open, high, low, close, adj_close, volume)
   - ~1.01M rows, 455 Indian stocks plus a synthetic 'NIFTY_50' ticker, 2014-2024
   - All prices are in Indian Rupees (INR).
2. stock_metadata (ticker, sector, market_cap_band) -- ticker info only, no dates
3. macro_indicators (date, fed_funds_rate) -- US Fed Funds Rate ONLY, no stock data
4. india_vix (date, vix_close) -- India VIX value ONLY, no stock data, no joins to daily_prices needed for VIX questions

CRITICAL date format note:
- Every table's date column is named exactly "date" (not vix_date, not price_date, etc).
- All date columns are stored as TEXT with a time component, e.g. '2020-03-23 00:00:00'.
- For an exact-date lookup, ALWAYS use: DATE(date) = '2020-03-23'
- NEVER compare the date column directly to a plain 'YYYY-MM-DD' string with =.

CRITICAL SQLite syntax rules:
- This is SQLite, NOT MySQL or PostgreSQL.
- There is NO YEAR(), MONTH(), or DATE_PART() function.
- To extract a year from a date, use: strftime('%Y', date) = '2023'

Rules:
- Generate ONLY a single valid SQLite query that answers the question.
- Return the query inside a ```sql code block, and nothing else -- no explanation.
- Use ONLY the exact column names listed above. Never invent a column name.
- A question about VIX needs ONLY the india_vix table -- do not join it to daily_prices.
- Limit results to 20 rows unless the question asks for more.
- Never use INSERT, UPDATE, DELETE, DROP, or ALTER -- only SELECT statements.
"""

In [ ]:
def ask_data_question(question):
    sql_prompt = f"{SIMPLE_SYSTEM_PROMPT}\n\nQuestion: {question}"
    raw_response = ask_local_llm(sql_prompt, max_new_tokens=200)

    sql = extract_sql(raw_response)
    if not sql:
        return {"question": question, "sql": None, "data": None,
                "interpretation": f"Could not extract a valid SQL query. Raw model output: {raw_response}"}

    if not is_safe_query(sql):
        return {"question": question, "sql": sql, "data": None,
                "interpretation": "Generated query contained a forbidden keyword and was blocked for safety."}

    try:
        result = pd.read_sql(sql, conn)
    except Exception as e:
        return {"question": question, "sql": sql, "data": None,
                "interpretation": f"Query execution failed: {e}"}

    if result.empty or (result.shape == (1, 1) and result.iloc[0, 0] is None):
        interpretation = "The query ran successfully but returned no usable result."
    else:
        interpretation_prompt = (
            f"Result table (this is the ONLY data you may reference):\n{result.to_string()}\n\n"
            f"Question: {question}\n\n"
            f"Write exactly one sentence stating the value(s) shown in the Result table above. "
            f"Do not invent any number not literally present in the Result table."
        )
        interpretation = ask_local_llm(interpretation_prompt, max_new_tokens=80)

    return {"question": question, "sql": sql, "data": result, "interpretation": interpretation}

def validate_columns_exist(sql, conn):
    try:
        pd.read_sql(f"EXPLAIN QUERY PLAN {sql}", conn)
        return True, None
    except Exception as e:
        return False, str(e)

In [ ]:
'''
test_questions = [
    "What is the average daily trading volume for RELIANCE in 2023?",
    "Which sector has the most stocks in the database?",
    "What was the highest closing price ever recorded for TCS?",
    "How many distinct stocks are in the Information Technology sector?",
    "What was the India VIX closing value on 2020-03-23?"
]

for q in test_questions:
    print(f"\n{'='*60}\nQuestion: {q}\n{'='*60}")
    result = ask_data_question(q)
    print(f"SQL: {result['sql']}")
    print(f"Data:\n{result['data']}")
    print(f"Interpretation: {result['interpretation']}")
'''

'\ntest_questions = [\n    "What is the average daily trading volume for RELIANCE in 2023?",\n    "Which sector has the most stocks in the database?",\n    "What was the highest closing price ever recorded for TCS?",\n    "How many distinct stocks are in the Information Technology sector?",\n    "What was the India VIX closing value on 2020-03-23?"\n]\n\nfor q in test_questions:\n    print(f"\n{\'=\'*60}\nQuestion: {q}\n{\'=\'*60}")\n    result = ask_data_question(q)\n    print(f"SQL: {result[\'sql\']}")\n    print(f"Data:\n{result[\'data\']}")\n    print(f"Interpretation: {result[\'interpretation\']}")\n'

In [ ]:
!pkill -f streamlit

In [ ]:
import subprocess
streamlit_process = subprocess.Popen(
    ["streamlit", "run", "/content/drive/MyDrive/Equity/app/chatbot.py",
     "--server.port", "8502", "--server.headless", "true"],
    stdout=open("/content/streamlit_log.txt", "w"),
    stderr=subprocess.STDOUT
)
print(f"Streamlit started, PID: {streamlit_process.pid}")

Streamlit started, PID: 3778


In [ ]:
!pip install -q pyngrok

In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token("3FISSemK7VRwDyWeKHqLRMH0Vk7_4S5rQjyuGwxw1MvQWLJz1")

public_url = ngrok.connect(8502)
print(f"Your app is live at: {public_url}")

Your app is live at: NgrokTunnel: "https://thesis-ouch-drew.ngrok-free.dev" -> "http://localhost:8502"


In [ ]:
'''


=========================================================================================================================================================
Chatbot.py
=========================================================================================================================================================

import os
import re
import sqlite3
import time

import pandas as pd
import streamlit as st
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

PROJECT_ROOT = "/content/drive/MyDrive/Equity"
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
DB_PATH = os.path.join(DATA_DIR, "equity_market.db")

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

SIMPLE_SYSTEM_PROMPT = """You are a financial data analyst assistant with access to a SQLite database.

Tables (use these EXACT column names, do not invent or rename any column):
1. daily_prices (ticker, date, open, high, low, close, adj_close, volume)
   - ~1.01M rows, 455 Indian stocks plus a synthetic 'NIFTY_50' ticker, 2014-2024
   - All prices are in Indian Rupees (INR).
2. stock_metadata (ticker, sector, market_cap_band) -- ticker info only, no dates
3. macro_indicators (date, fed_funds_rate) -- US Fed Funds Rate ONLY, no stock data
4. india_vix (date, vix_close) -- India VIX value ONLY, no stock data, no joins to daily_prices needed for VIX questions

CRITICAL date format note:
- Every table's date column is named exactly "date" (not vix_date, not price_date, etc).
- All date columns are stored as TEXT with a time component, e.g. '2020-03-23 00:00:00'.
- For an exact-date lookup, ALWAYS use: DATE(date) = '2020-03-23'
- NEVER compare the date column directly to a plain 'YYYY-MM-DD' string with =.

CRITICAL SQLite syntax rules:
- This is SQLite, NOT MySQL or PostgreSQL.
- There is NO YEAR(), MONTH(), or DATE_PART() function.
- To extract a year from a date, use: strftime('%Y', date) = '2023'

Rules:
- Generate ONLY a single valid SQLite query that answers the question.
- Return the query inside a ```sql code block, and nothing else -- no explanation.
- Use ONLY the exact column names listed above. Never invent a column name.
- A question about VIX needs ONLY the india_vix table -- do not join it to daily_prices.
- Limit results to 20 rows unless the question asks for more.
- Never use INSERT, UPDATE, DELETE, DROP, or ALTER -- only SELECT statements.
"""


@st.cache_resource
def load_model():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
    return tokenizer, model


@st.cache_resource
def get_connection():
    return sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True, check_same_thread=False)


def ask_local_llm(tokenizer, model, prompt, max_new_tokens=200):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.3, do_sample=True)
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response


def extract_sql(model_output):
    match = re.search(r"```sql\s*(.*?)```", model_output, re.DOTALL)
    if match:
        return match.group(1).strip()
    match = re.search(r"```\s*(.*?)```", model_output, re.DOTALL)
    if match:
        return match.group(1).strip()
    return None


def is_safe_query(sql):
    forbidden = ["insert", "update", "delete", "drop", "alter", "create", "attach", "pragma"]
    sql_lower = sql.lower()
    return not any(word in sql_lower for word in forbidden)


def ask_data_question(tokenizer, model, conn, question):
    sql_prompt = f"{SIMPLE_SYSTEM_PROMPT}\n\nQuestion: {question}"
    raw_response = ask_local_llm(tokenizer, model, sql_prompt, max_new_tokens=200)

    sql = extract_sql(raw_response)
    if not sql:
        return {
            "question": question,
            "sql": None,
            "data": None,
            "interpretation": f"Could not extract a valid SQL query from the model's output: {raw_response}",
        }

    if not is_safe_query(sql):
        return {
            "question": question,
            "sql": sql,
            "data": None,
            "interpretation": "Generated query contained a forbidden keyword and was blocked for safety.",
        }

    try:
        result = pd.read_sql(sql, conn)
    except Exception as e:
        return {"question": question, "sql": sql, "data": None, "interpretation": f"Query execution failed: {e}"}

    if result.empty or (result.shape == (1, 1) and result.iloc[0, 0] is None):
        interpretation = "The query ran successfully but returned no usable result."
    else:
        interpretation_prompt = (
            f"Result table (this is the ONLY data you may reference):\n{result.to_string()}\n\n"
            f"Question: {question}\n\n"
            f"Write exactly one sentence stating the value(s) shown in the Result table above. "
            f"Do not invent any number not literally present in the Result table."
        )
        interpretation = ask_local_llm(tokenizer, model, interpretation_prompt, max_new_tokens=80)

    return {"question": question, "sql": sql, "data": result, "interpretation": interpretation}


st.set_page_config(page_title="Equity Market Intelligence", layout="wide")

st.title("Equity Market Intelligence Platform")
st.caption("Ask a question about 455 Indian stocks, 2014\u20132024.")

if not os.path.exists(DB_PATH):
    st.error(f"Database not found at {DB_PATH}. Mount Google Drive and confirm the path before running this app.")
    st.stop()

with st.spinner("Loading local model (first run only, a few minutes)..."):
    tokenizer, model = load_model()

conn = get_connection()

with st.sidebar:
    st.subheader("About this database")
    st.write("455 NIFTY 500 stocks \u00b7 2014\u20132024 \u00b7 ~1.01M daily price records")
    st.write("Powered by a local open-source LLM (Qwen2.5-1.5B)")
    st.write("Note: This model reliably handles direct lookups and simple aggregations. Multi-step questions are not supported by it.")

    st.divider()

    st.subheader("Try one of these")
    example_questions = [
        "What is the average daily trading volume for RELIANCE in 2023?",
        "Which sector has the most stocks in the database?",
        "What was the highest closing price ever recorded for TCS?",
        "How many distinct stocks are in the Information Technology sector?",
    ]

    selected_example = None
    for q in example_questions:
        if st.button(q, use_container_width=True):
            selected_example = q

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        if msg["role"] == "assistant" and "sql" in msg:
            st.code(msg["sql"] if msg["sql"] else "No SQL generated", language="sql")
            if msg["data"] is not None and not msg["data"].empty:
                st.dataframe(msg["data"], use_container_width=True)
            st.write(msg["interpretation"])
        else:
            st.write(msg["content"])

prompt = st.chat_input("Ask about the market...")
if selected_example:
    prompt = selected_example

if prompt:
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.write(prompt)

    with st.chat_message("assistant"):
        with st.spinner("Generating SQL and running the query..."):
            result = ask_data_question(tokenizer, model, conn, prompt)

        st.code(result["sql"] if result["sql"] else "No SQL generated", language="sql")
        if result["data"] is not None and not result["data"].empty:
            st.dataframe(result["data"], use_container_width=True)
        st.write(result["interpretation"])

    st.session_state.messages.append(
        {
            "role": "assistant",
            "sql": result["sql"],
            "data": result["data"],
            "interpretation": result["interpretation"],
        }
    )

=========================================================================================================================================================

'''

<>:77: SyntaxWarning: invalid escape sequence '\s'
<>:77: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_2534/3375714621.py:77: SyntaxWarning: invalid escape sequence '\s'
  match = re.search(r"```sql\s*(.*?)```", model_output, re.DOTALL)


'\n\n\n=========================================================================================================================================================\nChatbot.py\n=========================================================================================================================================================\n\nimport os\nimport re\nimport sqlite3\nimport time\n\nimport pandas as pd\nimport streamlit as st\nimport torch\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\n\nPROJECT_ROOT = "/content/drive/MyDrive/Equity"\nDATA_DIR = os.path.join(PROJECT_ROOT, "data")\nDB_PATH = os.path.join(DATA_DIR, "equity_market.db")\n\nMODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"\n\nSIMPLE_SYSTEM_PROMPT = """You are a financial data analyst assistant with access to a SQLite database.\n\nTables (use these EXACT column names, do not invent or rename any column):\n1. daily_prices (ticker, date, open, high, low, close, adj_close, volume)\n   - ~1.01M rows, 455 Indian stocks plus